# 02 — MD production-config provenance (BayesOpt / Optuna)

> Read this NB if you want to know **which knobs the 270-set was built with** and who picked them. It just reads the vendored BayesOpt artifacts in `data/external/bayesopt/` — nothing is re-run here:
>
> - `bayes_opt.db` — Optuna SQLite study DB (studies `md_prod_v1`, `md_prod_v2`, plus a smoke)
> - `bo_winner_v1.json` — the top-1 config from `md_prod_v1` (trial 96, fitness 20.073)
> - `code/` — the 17-parameter search-space source (`space_v2.py` v2, `space.py` v1)
> - `report/BO_report.{ipynb,html}` — the reviewed BayesOpt report

The 270-set (9 targets × 30 ligands × 1 replica × 30 ns) was produced with the winner config in `bo_winner_v1.json`. The MD summary in `STUDY_DESIGN.md` ("30 ns AMBER-ff14SB / GAFF2 / TIP3P …") is that winner. It's not a hand-picked default.

**Preliminary.** BayesOpt v2 has 33 completed trials of ~200 planned; v1 has 200 complete. Numbers here are the v1 top-1.


> **Reader guide.** *Experiment A4 (see [STUDY_DESIGN §A4](../../STUDY_DESIGN.md)):* Bayesian
> optimisation of the production MDP — traceability of the winner back to a specific trial.
>
> **Question:** *which trial of the 200 + 34 BO trials produced the deployed production MDP,
> and what are its exact parameter values?*
>
> **Method:** SQL over the Optuna DB; per-parameter distribution + winner call-out.
>
> **Reproducibility contract:** reads `data/external/bayesopt/bayes_opt.db` (SQLite);
> winner MDP dumped to `data/external/bayesopt/bo_winner_v{1..5}.json`.

In [ ]:
NB_STEM = "50_md_config_provenance"
# ===== repo-relative setup — reruns from a fresh clone, no absolute paths =====
import sys, json, sqlite3
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").is_file()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from gbsabench.paths import FIGURES  # figures/upstream/
from gbsabench import style
style.apply_style()
NAVY, GOLD, GREY, GREYD, CREAM, WHITE = style.NAVY, style.GOLD, style.GREY, style.GREY_DASH, style.CREAM, style.WHITE

# Publication style — suppress in-figure titles (captured for CAPTIONS.md).
_SUPPRESSED_TITLES = []
def _capture_title(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
def _capture_suptitle(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
Axes.set_title  = _capture_title
Figure.suptitle = _capture_suptitle

# Capture every figure so the last cell can export them.
_CREATED_FIGS = []
if not getattr(plt.subplots, "_gbsa_wrapped", False):
    _orig_subplots, _orig_figure = plt.subplots, plt.figure
    def _register(fig):
        if not any(fig is seen for seen in _CREATED_FIGS):
            _CREATED_FIGS.append(fig)
        return fig
    def _cap_subplots(*a, **k):
        result = _orig_subplots(*a, **k); _register(result[0]); return result
    def _cap_figure(*a, **k):
        return _register(_orig_figure(*a, **k))
    _cap_subplots._gbsa_wrapped = _cap_figure._gbsa_wrapped = True
    plt.subplots, plt.figure = _cap_subplots, _cap_figure

def _figs():
    """Export every figure this notebook created to figures/upstream/."""
    FIGURES.mkdir(parents=True, exist_ok=True)
    for i, fig in enumerate(_CREATED_FIGS, start=1):
        fig.savefig(FIGURES / f"{NB_STEM}_fig{i}.png")
    print("exported", len(_CREATED_FIGS), "figure(s) to", FIGURES.relative_to(_root))

# --- BayesOpt artifact paths (all RELATIVE to repo root) --------------------
BO_ROOT   = _root / "data" / "external" / "bayesopt"
BO_DB     = BO_ROOT / "bayes_opt.db"
BO_WINNER = BO_ROOT / "bo_winner_v1.json"
BO_CODE   = BO_ROOT / "code"

assert BO_DB.is_file(), f"missing {BO_DB}"
assert BO_WINNER.is_file(), f"missing {BO_WINNER}"
print("BO artifacts:")
for p in [BO_DB, BO_WINNER, BO_CODE]:
    print(" -", p.relative_to(_root),
          f"({p.stat().st_size // 1024} KB)" if p.is_file() else "(dir)")


## How the 270 discovery-9 productions were configured

The MD production `.mdp` for the 270 discovery-9 runs is the **top-1 config from an Optuna Bayesian-optimisation (TPE) search** over a 17-parameter GROMACS-2025 MDP + FF space. The 17 axes (`data/external/bayesopt/code/space_v2.py`):

| Tier | Axis | Values |
|------|------|--------|
| 2 | `rcut` | 0.9, 1.0, 1.1 nm |
| 2 | `fourierspacing` | 0.10, 0.12, 0.14, 0.16 nm |
| 3 | `integrator` | `md`, `md-vv`, `sd` |
| 4 | `mts` (MTS = multi-timestep) | False, True (only when integrator=md) |
| 4 | `mts_level2_factor` | 2, 3, 4 (only when mts=True) |
| 5 | `nstlist` | 10, 20, 40, 80 (must divide `mts_level2_factor`) |
| 6 | `constraints` | `h-bonds`, `h-angles`, `all-bonds` |
| 6 | `hmr_factor` (HMR = hydrogen mass repartitioning) | 1.0, 2.5, 3.0, 3.5, 4.0 (1.0 = no HMR) |
| 7 | `lincs_order` (LINCS = linear constraint solver) | 4, 6, 8 |
| 7 | `lincs_iter` | 1, 2 |
| 8 | `dt` | 0.001, 0.002, 0.003, 0.004, 0.005 ps |
| 9 | `pcoupl` | `Parrinello-Rahman`, `C-rescale`, `Berendsen` (md/sd); `C-rescale`, `Berendsen` (md-vv) |
| 9 | `tcoupl` | `v-rescale`, `nose-hoover`, `berendsen` (`no` when integrator=sd; PR-safe pool when pcoupl=PR) |
| 10 | `tau_t` | 0.1, 0.5, 1.0, 2.0 ps |
| 10 | `tau_p` | 1.0, 2.0, 5.0, 10.0 ps (5, 10 only if pcoupl=Parrinello-Rahman) |
| 11 | `nstpcouple` | 10, 20 |
| 11 | `nstcalcenergy` | 50, 100 |

**Sampler and philosophy.** Optuna TPE (`optuna.samplers.TPESampler`), single MAXIMIZE objective: BO fitness = ns/day × stability weight. The `space_v2.py` docstring puts it plainly:

> *"if there are errors we report the errors, but skip nothing" → we re-open the whole envelope and let grompp / mdrun be the ground truth on what actually works. Failed trials record their `rc` and `stability_reason` in `trial_result.json`; nothing is preemptively gated out on empirical guesses about what "should" fail.*

Only *definitional* grompp-fatal combos are excluded up front (e.g. `integrator=md-vv + pcoupl=Parrinello-Rahman`). Empirically-doubtful combos (`dt=5 fs + HMR=1.0`, etc.) are proposed and **allowed to fail** — the failures are signal.

### FF-Tier-1 freeze — why FF is not in the search space

Force-field family is **frozen** across the whole study to `ff14SB + gaff2 + tip3p` with `vdw_modifier = Potential_Shift`. From `space_v2.py:95–98`:

```python
# === Tier 1: FF frozen (same as v1 — the seed topology is pre-equilibrated)
protein_ff = "ff14SB"
ligand_ff = "gaff2"
water_model = "tip3p"
```

The fuller rationale is in v1 `space.py:174–179`:

> BO trials re-run only md_production against a **cached, pre-equilibrated topology**, so the protein_ff / water_model / ligand_ff keys can't retroactively swap the FF — including them in the search space would just waste TPE budget on duplicate physics. Varying FF mid-flight against a topology that was minimised, heated and NPT-equilibrated under `ff14SB / gaff2 / tip3p` is not physically valid; it would restart the equilibration.

A full FF-comparison campaign (ff14SB vs ff19SB vs charmm36m, with matched water) needs a per-trial seed regenerate. That's a separate study for later.


In [ ]:
# --- Winner config (Optuna study `md_prod_v1`, trial 96, fitness 20.073) ----
winner = json.loads(BO_WINNER.read_text())
print("winner provenance:")
for k, v in winner["provenance"].items():
    print(f"  {k:24s} {v}")

# Flatten mdp + forcefield into a readable table.
rows = [("forcefield", k, v) for k, v in winner["forcefield"].items()]
rows += [("mdp",       k, v) for k, v in winner["mdp"].items()]
winner_tbl = pd.DataFrame(rows, columns=["section", "key", "value"])

# Highlight the non-obvious BO choices (all conservative envelope):
NONOBVIOUS = {"integrator": "md", "dt": 0.002, "tcoupl": "berendsen",
              "pcoupl": "C-rescale", "lincs_order": 4}
print("\n=== Non-obvious BO choices (the 'conservative envelope' the search found) ===")
for k, expected in NONOBVIOUS.items():
    got = winner["mdp"].get(k)
    tag = "OK" if got == expected else "!!"
    print(f"  [{tag}] {k:14s} = {got!r:>15s}  (expected {expected!r})")
print("\nAlso: HMR is OFF (no mass_repartition_factor key in v1 winner), MTS is OFF.\n")

winner_tbl


In [ ]:
# --- Enumerate studies + trial states from the Optuna SQLite DB -------------
# Optuna is not in the pixi env; we read the schema directly with sqlite3 stdlib.
con = sqlite3.connect(f"file:{BO_DB}?mode=ro", uri=True)

studies = pd.read_sql(
    "SELECT study_id, study_name FROM studies ORDER BY study_id", con)
print("studies in bayes_opt.db:")
print(studies.to_string(index=False))

directions = pd.read_sql(
    "SELECT study_id, direction FROM study_directions ORDER BY study_id", con)
print("\ndirections:")
print(directions.to_string(index=False))

counts = pd.read_sql("""
    SELECT s.study_name, t.state, COUNT(*) AS n_trials
    FROM trials t JOIN studies s USING(study_id)
    GROUP BY s.study_name, t.state
    ORDER BY s.study_name, t.state
""", con)
print("\ntrials per (study, state):")
print(counts.to_string(index=False))

best = pd.read_sql("""
    SELECT s.study_name,
           COUNT(v.value) AS n_scored,
           MIN(v.value) AS min_fitness,
           MAX(v.value) AS max_fitness
    FROM trial_values v
      JOIN trials t USING(trial_id)
      JOIN studies s USING(study_id)
    GROUP BY s.study_name
    ORDER BY s.study_name
""", con)
print("\nfitness range per study (MAXIMIZE direction, higher = faster/stabler):")
print(best.to_string(index=False))

# Verify the winner value 20.073 is findable in md_prod_v1
winner_target = float(winner["provenance"]["stage1_fitness"])
winner_row = pd.read_sql(f"""
    SELECT t.number AS trial_number, v.value AS fitness
    FROM trials t
      JOIN trial_values v USING(trial_id)
      JOIN studies s USING(study_id)
    WHERE s.study_name = 'md_prod_v1'
    ORDER BY v.value DESC LIMIT 5
""", con)
print("\ntop-5 md_prod_v1 trials by fitness:")
print(winner_row.to_string(index=False))
top1 = winner_row.iloc[0]
match = abs(top1["fitness"] - winner_target) < 1e-3
print(f"\nwinner-json fitness {winner_target}  vs  DB best {top1['fitness']:.6f}  "
      f"trial {int(top1['trial_number'])}  match={match}")
assert match, "Winner value in JSON does not match DB — investigate."


In [ ]:
# --- Figure 1: fitness vs trial number for md_prod_v1 -----------------------
v1 = pd.read_sql("""
    SELECT t.number AS trial, v.value AS fitness
    FROM trials t
      JOIN trial_values v USING(trial_id)
      JOIN studies s USING(study_id)
    WHERE s.study_name = 'md_prod_v1' AND t.state = 'COMPLETE'
    ORDER BY t.number
""", con)
print(f"md_prod_v1: {len(v1)} completed trials")

v1["best_so_far"] = v1["fitness"].cummax()

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.scatter(v1["trial"], v1["fitness"], s=22, color=NAVY, alpha=0.65,
           edgecolor="none", label="trial fitness")
ax.plot(v1["trial"], v1["best_so_far"], color=GOLD, lw=2.2,
        label="best so far")
best_i = v1["fitness"].idxmax()
ax.scatter([v1.loc[best_i, "trial"]], [v1.loc[best_i, "fitness"]],
           s=110, color=GOLD, edgecolor=NAVY, lw=1.4, zorder=5,
           label=f"winner (trial {int(v1.loc[best_i,'trial'])}, "
                 f"fitness {v1.loc[best_i,'fitness']:.3f})")
ax.set_xlabel("Optuna trial number")
ax.set_ylabel("fitness  (ns/day × stability)")
ax.set_title("md_prod_v1 fitness vs trial (200 completed trials)")
ax.legend(loc="lower right")
plt.tight_layout()


In [ ]:
# --- Figure 2: which parameter values dominate the top-10 md_prod_v1 trials --
params = pd.read_sql("""
    SELECT t.number AS trial, v.value AS fitness,
           p.param_name, p.param_value, p.distribution_json
    FROM trials t
      JOIN trial_values v USING(trial_id)
      JOIN trial_params p USING(trial_id)
      JOIN studies s USING(study_id)
    WHERE s.study_name = 'md_prod_v1' AND t.state = 'COMPLETE'
""", con)
print(f"md_prod_v1 param rows: {len(params)}   unique params: {params.param_name.nunique()}")

# Decode categorical params (stored as float indices into choices JSON).
import ast
def _decode(row):
    dj = json.loads(row["distribution_json"])
    attrs = dj.get("attributes", {})
    choices = attrs.get("choices")
    if choices is not None:  # CategoricalDistribution
        try:
            return choices[int(row["param_value"])]
        except (ValueError, IndexError, TypeError):
            return row["param_value"]
    return row["param_value"]

params["decoded"] = params.apply(_decode, axis=1)

# Top-10 trials by fitness
top10 = params.sort_values("fitness", ascending=False).drop_duplicates("trial").head(10)["trial"].tolist()
top_params = params[params["trial"].isin(top10)]
param_names = sorted(top_params["param_name"].unique())
print(f"top-10 trials: {top10}")
print(f"param axes shown: {len(param_names)}")

# Compact grid — up to 17 subplots.
ncols = 4
nrows = (len(param_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 2.3 * nrows))
axes = np.asarray(axes).reshape(-1)
for ax, pname in zip(axes, param_names):
    sub = top_params[top_params["param_name"] == pname]
    counts = sub["decoded"].astype(str).value_counts().sort_index()
    ax.bar(range(len(counts)), counts.values, color=NAVY, edgecolor=GOLD, lw=1.0)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index.tolist(), rotation=35, ha="right", fontsize=8)
    ax.set_ylabel("hits in top-10")
    ax.set_title(pname, fontsize=9)
    ax.tick_params(axis="y", labelsize=8)
for ax in axes[len(param_names):]:
    ax.set_visible(False)
plt.suptitle("Parameter values in the top-10 md_prod_v1 trials (which corner won)")
plt.tight_layout(rect=[0, 0, 1, 0.97])


## Three-tier MD data structure

The MD data across the discovery-9 program falls into three tiers. This repo analyses **Tier 1**. The other two are noted here for context.

| Tier | What | Where | Purpose | Analysed here? |
|------|------|-------|---------|----------------|
| 1 | 270-set (9 targets × 30 ligands × 1 replica × **30 ns**) | `data/raw/complex_analyses/` | Production. Downstream MD × BEDROC analysis. | **Yes** — every downstream NB. |
| 2 | BO trial productions (200 v1 + 34 v2 = 234 trials × ~20 ps each) | `data/external/bayesopt/bayes_opt.db` | BayesOpt search history (per-trial config + fitness + failure reason). | This NB reads the DB. |
| 3 | Winner-replica workspace (8 targets × 30 ligands × **3 replicas × 20 ns**) | `/mnt/netapp1/Store_othcxlwa/pipeline-mps-workspaces/discovery9_winner/` | Confirmatory production with the winner config, in triplicate. Planned. | **No** — data not yet copied to this repo. |

Note: Tier 2 trials are short (~20 ps). They're timing/stability probes, not production trajectories. Trial length is env-tunable via `PIPELINE_MPS_BO_PS` (default 20 000 ps = 20 ns per trial; the shipped campaigns used shorter budgets for TPE efficiency). Tier 3 isn't in this repo's `data/` tree yet.


## How to re-open the BO in Optuna

If you have `optuna` in your environment (this repo's pixi env doesn't by default — pin it in `env/pixi.toml` if you want it), attach to the vendored study read-only:

```python
import optuna
from pathlib import Path

db = Path("data/external/bayesopt/bayes_opt.db").resolve()
study = optuna.load_study(
    study_name="md_prod_v1",
    storage=f"sqlite:///{db}",
)

print(f"n_trials={len(study.trials)}  best_value={study.best_value:.3f}")
print(study.best_params)

# The v2 study is under study_name='md_prod_v2'. There is also a
# 'smoke_5hu9_ligand01_v1' 1-trial smoke study.

# For plots without launching Optuna Dashboard:
import optuna.visualization.matplotlib as ovm
ovm.plot_optimization_history(study)
ovm.plot_param_importances(study)
```

Or point Optuna Dashboard at the same file:

```bash
pip install optuna-dashboard
optuna-dashboard sqlite:///data/external/bayesopt/bayes_opt.db
```

Opening the DB read-only via `sqlite:///…?mode=ro` does not write to it. This notebook does exactly that in Cell 5.


In [ ]:
# Close the DB and export every figure this notebook created.
con.close()
_figs()
